In [1]:
print("a")

a


In [ ]:
import numpy as np
from scipy.special import erfc
from itertools import product
 
def Fhit_function(radius, distance, diffusionCoef, t):
    """Calculates the cumulative hitting probability up to time t."""
    if t <= 0:
        return 0.0
    return (radius / (distance + radius)) * erfc(distance / np.sqrt(4 * diffusionCoef * t))
 
def calculate_hitting_probabilities(mem_len, radius, distance, diffusionCoef, Ts):
    """Calculates interval hitting probabilities P[i] for each symbol slot."""
    P = np.zeros(mem_len)
    for i in range(mem_len):
        t_end = (i + 1) * Ts
        t_start = i * Ts
        P[i] = Fhit_function(radius, distance, diffusionCoef, t_end) - Fhit_function(radius, distance, diffusionCoef, t_start)
    return P
 
def bit_error_rate(mem_len, threshold, N, P):
    """
    Calculates the Bit Error Rate (BER) accounting for Inter-Symbol Interference (ISI)
    using a Gaussian approximation with precomputed channel coefficients P[i].
   
    Parameters:
    - mem_len: Total length of the bit sequence to consider (current bit + previous bits)
    - threshold: The decision threshold (number of molecules)
    - N: Number of molecules released for bit '1'
    - P: Interval hitting probabilities where P[0] is for [0, Ts], P[1] is for [Ts, 2Ts], etc.
    """
    P = np.asarray(P)
    if len(P) < mem_len:
        raise ValueError("Length of P must be at least mem_len.")

    total_ber = 0.0
    num_sequences = 2**mem_len
   
    # 1. Generate all possible bit sequences of length mem_len
    sequences = list(product([0, 1], repeat=mem_len))
   
    for seq in sequences:
        # Reverse sequence so seq_rev[0] is the current bit, seq_rev[1] is the previous bit, etc.
        seq_rev = seq[::-1]
        current_bit = seq_rev[0]
       
        # 2. Calculate Mean and Variance for this specific sequence
        mu = 0.0
        variance = 0.0
        for i in range(mem_len):
            bit = seq_rev[i]
            mu += bit * N * P[i]
            # Variance of a sum of independent Binomials: np(1-p)
            variance += bit * N * P[i] * (1 - P[i])
           
        std = np.sqrt(variance)
       
        # 3. Calculate conditional error probability using Gaussian Q-function equivalent
        # Q(x) = 0.5 * erfc(x / sqrt(2))
        if std == 0:
            # Edge case: if sequence is all zeros, there are no molecules, hence no variance.
            if current_bit == 1:
                pe = 1.0 if mu < threshold else 0.0
            else:
                pe = 1.0 if mu >= threshold else 0.0
        else:
            if current_bit == 1:
                # True bit is 1, error if received signal < threshold
                pe = 0.5 * erfc((mu - threshold) / (std * np.sqrt(2)))
            else:
                # True bit is 0, error if received signal > threshold
                pe = 0.5 * erfc((threshold - mu) / (std * np.sqrt(2)))
               
        total_ber += pe
       
    # 4. Average the error probabilities across all possible sequences
    average_ber = total_ber / num_sequences
   
    return average_ber
 
# --- Example Usage ---
# Assuming your physical parameters from the previous script:
p_radius = 5
p_distance = 12.5 - p_radius  # Distance from point source to receiver surface
p_diffusionCoef = 79.4
N_molecules = 10**6
 
# New digital communication parameters
symbol_duration = 1  # Ts in seconds
decision_threshold = 20000  # Arbitrary threshold for decoding
memory_length = 5  # 5-bit sequence (current bit + 4 previous bits)
 
P = calculate_hitting_probabilities(
    mem_len=memory_length,
    radius=p_radius,
    distance=p_distance,
    diffusionCoef=p_diffusionCoef,
    Ts=symbol_duration
)

ber = bit_error_rate(
    mem_len=memory_length,
    threshold=decision_threshold,
    N=N_molecules,
    P=P
)
 
print(f"Calculated BER for memory length {memory_length}: {ber}")

In [ ]:
# Parameter lists for simulation sweep
import pandas as pd
from itertools import product

# Define parameter ranges
param_ranges = {
    'p_radius': [3, 5, 7],
    'p_distance': [7.5, 12.5, 15],  # Will calculate based on radius
    'p_diffusionCoef': [50, 79.4, 100],
    'N_molecules': [1e5, 1e6, 1e7],
    'symbol_duration': [0.5, 1, 2],
    'decision_threshold': [10000, 20000, 50000],
    'memory_length': [3, 5, 7]
}

# Create a list of parameter combinations
# For now, let's create a simpler approach: individual parameter lists
p_radius_list = [3, 5, 7]
p_diffusionCoef_list = [50, 79.4, 100]
N_molecules_list = [1e5, 1e6, 1e7]
symbol_duration_list = [0.5, 1, 2]
decision_threshold_list = [10000, 20000, 50000]
memory_length_list = [3, 5, 7]

# Create results storage
results = []

print("Parameter lists defined:")
print(f"  p_radius: {p_radius_list}")
print(f"  p_diffusionCoef: {p_diffusionCoef_list}")
print(f"  N_molecules: {N_molecules_list}")
print(f"  symbol_duration: {symbol_duration_list}")
print(f"  decision_threshold: {decision_threshold_list}")
print(f"  memory_length: {memory_length_list}")

In [ ]:
# Run simulations with parameter sweep
import time

total_simulations = (len(p_radius_list) * len(p_diffusionCoef_list) * 
                    len(N_molecules_list) * len(symbol_duration_list) * 
                    len(decision_threshold_list) * len(memory_length_list))

print(f"Total simulations to run: {total_simulations}")
print("Starting parameter sweep...")

start_time = time.time()
sim_count = 0

# Iterate through all parameter combinations
for radius in p_radius_list:
    for diffusion in p_diffusionCoef_list:
        for n_mol in N_molecules_list:
            for symbol_dur in symbol_duration_list:
                for threshold in decision_threshold_list:
                    for mem_len in memory_length_list:
                        # Calculate distance from radius
                        distance = 12.5 - radius
                        
                        # Precompute channel coefficients once for this parameter tuple
                        P = calculate_hitting_probabilities(
                            mem_len=mem_len,
                            radius=radius,
                            distance=distance,
                            diffusionCoef=diffusion,
                            Ts=symbol_dur
                        )
                        
                        # Run BER calculation
                        try:
                            ber = bit_error_rate(
                                mem_len=mem_len,
                                threshold=int(threshold),
                                N=int(n_mol),
                                P=P
                            )
                            
                            # Store results
                            results.append({
                                'p_radius': radius,
                                'p_distance': distance,
                                'p_diffusionCoef': diffusion,
                                'N_molecules': int(n_mol),
                                'symbol_duration': symbol_dur,
                                'decision_threshold': int(threshold),
                                'memory_length': mem_len,
                                'BER': ber
                            })
                            
                            sim_count += 1
                            if sim_count % 10 == 0:
                                print(f"  Completed {sim_count}/{total_simulations} simulations...")
                        
                        except Exception as e:
                            print(f"Error at iteration {sim_count}: {e}")
                            continue

elapsed_time = time.time() - start_time
print(f"\nCompleted {sim_count} simulations in {elapsed_time:.2f} seconds")
print(f"Average time per simulation: {elapsed_time/sim_count:.4f} seconds")

In [ ]:
# Convert results to DataFrame and save
df_results = pd.DataFrame(results)

print(f"Results DataFrame shape: {df_results.shape}")
print("\nFirst few rows:")
print(df_results.head(10))

print("\nResults summary statistics:")
print(df_results[['BER']].describe())

# Save to CSV
output_file = 'ber_simulation_results.csv'
df_results.to_csv(output_file, index=False)
print(f"\nResults saved to: {output_file}")

In [ ]:
# Analysis and filtering examples

# Find the best (lowest) BER
best_result = df_results.loc[df_results['BER'].idxmin()]
print("Best Configuration (Lowest BER):")
print(best_result)
print(f"\nBER: {best_result['BER']:.6e}\n")

# Filter results by criteria
print("=" * 60)
print("Example filters:\n")

# Results with BER < 0.001
low_ber = df_results[df_results['BER'] < 0.001]
print(f"Configurations with BER < 0.001: {len(low_ber)}")

# Results for specific memory length
mem_len_5 = df_results[df_results['memory_length'] == 5]
print(f"Configurations with memory_length=5: {len(mem_len_5)}")

# Group by memory length and show average BER
print("\nAverage BER by Memory Length:")
print(df_results.groupby('memory_length')['BER'].agg(['mean', 'min', 'max']))

print("\nAverage BER by Diffusion Coefficient:")
print(df_results.groupby('p_diffusionCoef')['BER'].agg(['mean', 'min', 'max']))

print("\nAverage BER by Decision Threshold:")
print(df_results.groupby('decision_threshold')['BER'].agg(['mean', 'min', 'max']))

In [ ]:
import plotly.graph_objects as go
import numpy as np

def generate_random_params(seed):
    """
    Generates a reproducible dictionary of physical and system parameters 
    for the molecular communication simulation using a random seed.
    """
    np.random.seed(seed)
    
    params = {
        # Integers
        'mem_len': np.random.randint(low=3, high=7),          # 3, 4, 5, or 6
        #'N': int(10**np.random.uniform(low=5, high=6.5)),     # 100,000 to ~3,160,000
        'N' : 10**3,  # Fixed N for consistent threshold sweep
        # Floats
        'radius': np.random.uniform(low=3.0, high=7.0),
        'distance': np.random.uniform(low=5.0, high=15.0),
        'diffusionCoef': np.random.uniform(low=50.0, high=100.0),
        'Ts': np.random.uniform(low=0.5, high=2.0)
    }
    
    return params

# 1. Define the seeds you want to test
seeds_to_test = [50]

fig = go.Figure()

for seed in seeds_to_test:
    # 2. Generate the parameters for this specific seed
    params = generate_random_params(seed)

    # 3. Precompute channel coefficients once for this seed-specific setup
    P = calculate_hitting_probabilities(
        mem_len=params['mem_len'],
        radius=params['radius'],
        distance=params['distance'],
        diffusionCoef=params['diffusionCoef'],
        Ts=params['Ts']
    )
    
    # 4. Create a dynamic sweep range based on this configuration's N
    # We use 500 points to keep the plot responsive while maintaining the "stepped" detail
    thresholds_sweep = np.linspace(1, params['N'], 500)
    #thresholds_sweep = np.linspace(110000, 550000, 1000)
    ber_values = []
    
    print(f"Calculating sweep for Seed {seed} (N={params['N']}, dist={params['distance']:.2f}, radius={params['radius']:.2f}, diffusion={params['diffusionCoef']:.2f}, Ts={params['Ts']:.2f}, mem_len={params['mem_len']})...") 
    
    # 5. Calculate BER
    for th in thresholds_sweep:
        ber = bit_error_rate(
            mem_len=params['mem_len'],
            threshold=th,
            N=params['N'],
            P=P
        )
        ber_values.append(ber)
        
    # 6. Find the empirical minimum to mark on the graph
    min_idx = np.argmin(ber_values)
    optimal_th = thresholds_sweep[min_idx]
    min_ber = ber_values[min_idx]
    
    # 7. Add the curve to the Plotly figure
    curve_name = f"Seed {seed} (λ*: {optimal_th:.0f})"
    fig.add_trace(go.Scatter(
        x=thresholds_sweep, 
        y=ber_values,
        mode='lines',
        name=curve_name
    ))
    
    # 8. Add a marker for the minimum point of this specific curve
    fig.add_trace(go.Scatter(
        x=[optimal_th], 
        y=[min_ber],
        mode='markers',
        name=f"Min Seed {seed}",
        marker=dict(size=10, symbol='x'),
        showlegend=False, # Hide the marker from the legend to keep it clean
        hovertemplate=f"Seed {seed} Min<br>λ*: {optimal_th:.0f}<br>BER: {min_ber:.2e}<extra></extra>"
    ))

# 9. Update layout for log scale and styling
fig.update_layout(
    title='BER vs. Decision Threshold (λ) Across Different Random Seeds',
    xaxis_title='Decision Threshold (λ) [Number of Molecules]',
    yaxis_title='Bit Error Rate (BER)',
    yaxis_type='log',  
    template='plotly_white',
    hovermode='x unified',
    width=1000,
    height=600
)

fig.show()

In [ ]:
import numpy as np
import time
from itertools import product as iproduct

# ----- Tunables --------------------------------------------------------------
MAX_MEM_LEN = 14       # hard cap for feasibility: 2^14 = 16384 sequences
ARRIVAL_COVERAGE = 0.70
N_SAMPLES = 20
SAVE_CSV = False
CSV_PATH = "training_data_random_channels.csv"

rng = np.random.default_rng(seed=7)


def F_hit_total(radius, distance):
    """Maximum cumulative hitting probability: F_hit(inf) = r / (r + d)."""
    return radius / (radius + distance)


def auto_mem_len_and_P(radius, distance, diffusionCoef, Ts,
                       coverage=ARRIVAL_COVERAGE, max_len=MAX_MEM_LEN):
    """
    Find a good memory length from channel physics:
    choose smallest mem_len with cumulative arrivals >= coverage * F_hit(inf),
    then return P[0..mem_len] (includes the extra P[mem_len] tap).
    """
    target = coverage * F_hit_total(radius, distance)
    cumsum = 0.0
    k = 0
    while k < max_len:
        p_k = (
            Fhit_function(radius, distance, diffusionCoef, (k + 1) * Ts)
            - Fhit_function(radius, distance, diffusionCoef, k * Ts)
        )
        cumsum += p_k
        k += 1
        if cumsum >= target:
            break

    # return mem_len+1 taps so P[mem_len] is available as requested
    P_ext = calculate_hitting_probabilities(k + 1, radius, distance, diffusionCoef, Ts)
    return k, P_ext


def ber_for_fixed_taps(mem_len, threshold, N, P):
    """Compute BER for a given channel taps."""
    P_arr = np.asarray(P, dtype=float)[:mem_len]

    # all 2^mem_len bit patterns, reversed so column 0 = current bit
    seqs = np.array(list(iproduct([0, 1], repeat=mem_len)), dtype=np.float64)[:, ::-1]
    c_bit = seqs[:, 0]

    mu = (seqs * (N * P_arr)).sum(axis=1)
    var = (seqs * (N * P_arr * (1.0 - P_arr))).sum(axis=1)
    std = np.sqrt(np.maximum(var, 0.0))

    pe = np.empty_like(mu)
    zero_std = std == 0

    # deterministic edge case
    pe[zero_std & (c_bit == 1)] = np.where(mu[zero_std & (c_bit == 1)] < threshold, 1.0, 0.0)
    pe[zero_std & (c_bit == 0)] = np.where(mu[zero_std & (c_bit == 0)] >= threshold, 1.0, 0.0)

    # gaussian approx case
    nz = ~zero_std
    pe[nz & (c_bit == 1)] = 0.5 * erfc((mu[nz & (c_bit == 1)] - threshold) / (std[nz & (c_bit == 1)] * np.sqrt(2)))
    pe[nz & (c_bit == 0)] = 0.5 * erfc((threshold - mu[nz & (c_bit == 0)]) / (std[nz & (c_bit == 0)] * np.sqrt(2)))

    return float(pe.mean())


SEP = "=" * 72
print(SEP)
print(f"Data generation with random channels (samples={N_SAMPLES})")
print(f"mem_len rule: cumulative arrivals >= {ARRIVAL_COVERAGE*100:.0f}% of F_hit(inf)")
print(f"mem_len cap: {MAX_MEM_LEN} (2^{MAX_MEM_LEN}={2**MAX_MEM_LEN:,} sequences)")
print(SEP)

rows = []

for s in range(N_SAMPLES):
    # Random channel + system inputs
    radius = rng.uniform(3.0, 7.0)
    distance = rng.uniform(3.0, 15.0)
    diffusionCoef = rng.uniform(50.0, 120.0)
    Ts = rng.uniform(0.5, 3.0)
    N = int(10 ** rng.uniform(3.0, 6.0))

    t0 = time.perf_counter()

    # Good mem_len from channel coefficients and 70% rule
    mem_len, P_ext = auto_mem_len_and_P(radius, distance, diffusionCoef, Ts)
    P_main = P_ext[:mem_len]
    P_extra = float(P_ext[mem_len])

    # Sample threshold (lambda) from a broad practical range
    # around expected first-slot mean arrivals.
    lambda_max = max(1.0, 1.5 * N * P_main[0])
    threshold = float(rng.uniform(1.0, lambda_max))

    # Output BER for this given input tuple
    BER = ber_for_fixed_taps(mem_len, threshold, N, P_ext)
    elapsed_ms = (time.perf_counter() - t0) * 1e3

    # Print requested input/output information
    print(f"\n-- Sample {s+1} " + "-" * 58)
    print(f"Channel: r={radius:.3f}, d={distance:.3f}, D={diffusionCoef:.3f}, Ts={Ts:.3f}")
    print(f"Inputs : N={N}, mem_len={mem_len}, lambda(threshold)={threshold:.3f}")
    print(f"{'i':<5} {'P[i]':>14} {'P[i]*(1-P[i])':>18}")
    print(f"{'-'*5} {'-'*14} {'-'*18}")
    for i, p in enumerate(P_main):
        print(f"{i:<5d} {p:>14.8f} {p*(1.0-p):>18.8f}")
    print(f"{'-'*5} {'-'*14} {'-'*18}")
    print(f"extra P[mem_len] = {P_extra:.8f} ; extra P*(1-P) = {P_extra*(1.0-P_extra):.8f}")
    print(f"Output : BER={BER:.6e} ({elapsed_ms:.2f} ms)")

    # Optional row format for model training
    rows.append({
        "radius": radius,
        "distance": distance,
        "diffusionCoef": diffusionCoef,
        "Ts": Ts,
        "N": N,
        "mem_len": mem_len,
        "threshold": threshold,
        "P_values": P_main.tolist(),
        "P_var_terms": (P_main * (1.0 - P_main)).tolist(),
        "P_mem_len_extra": P_extra,
        "P_mem_len_extra_var": P_extra * (1.0 - P_extra),
        "BER": BER,
        "elapsed_ms": elapsed_ms,
    })

print("\n" + SEP)
print("Done.")

if SAVE_CSV:
    import pandas as pd

    # Keep list-valued columns as strings in CSV; great for quick inspection.
    df_train = pd.DataFrame(rows)
    df_train.to_csv(CSV_PATH, index=False)
    print(f"Saved dataset to {CSV_PATH}")

In [ ]:
import numpy as np
from itertools import product as iproduct
from scipy.special import erfc

# Task 3: Fully random synthetic data (no smart/physics-based choices)
N_RANDOM_SAMPLES = 10
rng_rand = np.random.default_rng(seed=123)


def random_monotone_decreasing_P(mem_len, rng):
    """Generate a random monotone-decreasing P[i] sequence in (0, 1)."""
    p0 = float(rng.uniform(0.05, 0.95))
    dec_factors = rng.uniform(0.35, 0.98, size=mem_len - 1)
    P = np.empty(mem_len, dtype=float)
    P[0] = p0
    for i in range(1, mem_len):
        P[i] = P[i - 1] * dec_factors[i - 1]
    # Clamp away from exact 0/1 for numerical stability
    return np.clip(P, 1e-6, 1 - 1e-6)


def random_variances_for_P(P, rng):
    """
    Generate random variances for each tap, independent from P(1-P) rule.
    """
    mem_len = len(P)
    # Random positive variances proportional to the tap scale
    scales = rng.uniform(0.05, 1.00, size=mem_len)
    variances = scales * P
    return np.maximum(variances, 1e-9)


def bit_error_rate_with_custom_variances(mem_len, threshold, P, variances):
    """BER using provided P and provided per-tap variances (no N)."""
    P = np.asarray(P, dtype=float)
    variances = np.asarray(variances, dtype=float)

    if len(P) < mem_len or len(variances) < mem_len:
        raise ValueError("P and variances must have length at least mem_len.")

    seqs = np.array(list(iproduct([0, 1], repeat=mem_len)), dtype=np.float64)[:, ::-1]
    current_bits = seqs[:, 0]

    # Mean from P only; total variance from custom per-tap variances
    mu = (seqs * P[:mem_len]).sum(axis=1)
    var_total = (seqs * variances[:mem_len]).sum(axis=1)
    std = np.sqrt(np.maximum(var_total, 0.0))

    pe = np.empty_like(mu)
    zero_std = std == 0

    pe[zero_std & (current_bits == 1)] = np.where(
        mu[zero_std & (current_bits == 1)] < threshold, 1.0, 0.0
    )
    pe[zero_std & (current_bits == 0)] = np.where(
        mu[zero_std & (current_bits == 0)] >= threshold, 1.0, 0.0
    )

    nz = ~zero_std
    pe[nz & (current_bits == 1)] = 0.5 * erfc(
        (mu[nz & (current_bits == 1)] - threshold)
        / (std[nz & (current_bits == 1)] * np.sqrt(2))
    )
    pe[nz & (current_bits == 0)] = 0.5 * erfc(
        (threshold - mu[nz & (current_bits == 0)])
        / (std[nz & (current_bits == 0)] * np.sqrt(2))
    )

    return float(pe.mean())


print("=" * 72)
print("Task 3: Fully random synthetic BER data")
print("- Random monotone-decreasing P[i]")
print("- Random per-tap variances")
print("- Random threshold")
print("=" * 72)

for s in range(N_RANDOM_SAMPLES):
    mem_len = int(rng_rand.integers(3, 10))

    P = random_monotone_decreasing_P(mem_len, rng_rand)
    variances = random_variances_for_P(P, rng_rand)

    # Random threshold in a broad random range based on P only
    threshold = float(rng_rand.uniform(1e-3, max(2e-3, 2.0 * P[0])))

    ber = bit_error_rate_with_custom_variances(
        mem_len=mem_len,
        threshold=threshold,
        P=P,
        variances=variances,
    )

    print(f"\n-- Sample {s+1} " + "-" * 58)
    print(f"threshold = {threshold:.6f}")
    print(f"{'i':<5} {'P[i]':>14} {'variance[i]':>16}")
    print(f"{'-'*5} {'-'*14} {'-'*16}")
    for i in range(mem_len):
        print(f"{i:<5d} {P[i]:>14.8f} {variances[i]:>16.8f}")
    print(f"Output BER = {ber:.6e}")

print("\nDone.")